# Corpus Expansion — Long-Range Variant on Llama-3.1-8B

**Purpose:** test whether the order-specific marginal-context-benefit signal is genuinely heavy-tailed (continues past d ≈ 100) or has a cutoff at some characteristic distance.

**Method:** measure ordered + shuffled perplexity at log-spaced context lengths up to **1024 tokens** instead of dense 0–100. Same documents, same target positions (by fraction), much wider distance range.

**Sample sizes:** per-target compute is ~10× the dense run because the longest forward pass is now O(1024²) instead of O(100²), but we trade dense sampling (100 measurement points) for log-spaced (12 points), so total per-target time stays manageable. ~2-3 hours per cell on T4.

**Output:** `My Drive/LRTIA/Results/corpus_expansion_longrange/llama/<corpus_id>.json`. Different format from the dense run — has `context_lengths` listing the sampled distances and `ordered_ppl`, `shuffled_ppl` arrays of equal length.

**Min doc length filter:** texts must be at least 1024 + 30 = 1054 tokens.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time
from pathlib import Path
from scipy import stats
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/LRTIA')
BASE = DRIVE / 'Results/corpus_expansion_longrange/llama'
BASE.mkdir(parents=True, exist_ok=True)
TARGETS_PATH = DRIVE / 'Results/corpus_expansion/targets_llama.jsonl'
TOK_MANIFEST_PATH = DRIVE / 'Results/corpus_expansion/tokenized_manifest_llama.jsonl'

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'

# Log-spaced context lengths up to 1024.
CTX_LENGTHS = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
MAX_CTX = max(CTX_LENGTHS)
TARGET_LEN = 30
TARGET_FRACS = [0.5]   # one target per doc to keep runtime bounded; can re-add 0.25/0.75 later
MIN_DOC_TOK = MAX_CTX + TARGET_LEN + 50  # safety margin
N_SHUFFLES = 1
SEED = 20260503

# Phase 1 (already cached): gutenberg_fiction_en, ted_transcripts_en, ted_transcripts_de, literary_ja, literary_fi.
# Phase 2 expansion to match the broader paper claim:
#   - news_en               written expository (English news)
#   - ted_transcripts_fr    prepared spoken, IE Romance
#   - ted_transcripts_tr    prepared spoken, non-IE (Turkic, agglutinative)
#   - buckeye               spontaneous spoken English (uses jsonl loader below)
RUN_CORPORA = [
    'gutenberg_fiction_en',
    'ted_transcripts_en',
    'ted_transcripts_de',
    'literary_ja',
    'literary_fi',
    'news_en',
    'ted_transcripts_fr',
    'ted_transcripts_tr',
    'buckeye',
]

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Probe: {MODEL_NAME}')
print(f'Context lengths: {CTX_LENGTHS}')
print(f'Min doc length: {MIN_DOC_TOK} tokens')
print(f'Cells: {RUN_CORPORA}')

In [ ]:
# Load model — fp16 for H100 (skip 4-bit quant; dequant overhead hurts on H100).
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print(f'{MODEL_NAME} loaded (fp16)')

In [ ]:
# Pipeline: ppl_nll same as dense version.
@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2:
        return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i + 1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf'), float('inf')
    mn = nll / cnt
    return math.exp(mn), mn

# Long-range curves: only at the log-spaced CTX_LENGTHS, not dense.
# Returns dict with context_lengths, ordered_ppl, shuffled_ppl arrays of equal length.
def compute_longrange_curves(full_ids, target_start, target_end):
    tgt = full_ids[target_start:target_end]
    o_ppl, s_ppl = [], []
    for c in CTX_LENGTHS:
        if c == 0:
            pfx = []
        else:
            pfx = full_ids[target_start - c:target_start]
        # Ordered
        p_ord, _ = ppl_nll(pfx, tgt)
        o_ppl.append(p_ord)
        if c == 0:
            s_ppl.append(p_ord)   # no order to shuffle at c=0
        else:
            rng = random.Random(SEED + c)
            sh_ppls = []
            for _ in range(N_SHUFFLES):
                sh = list(pfx); rng.shuffle(sh)
                p_sh, _ = ppl_nll(sh, tgt)
                if not math.isinf(p_sh): sh_ppls.append(p_sh)
            s_ppl.append(np.mean(sh_ppls) if sh_ppls else p_ord)
    return {
        'context_lengths': list(CTX_LENGTHS),
        'ordered_ppl': o_ppl,
        'shuffled_ppl': s_ppl,
    }

print('Pipeline ready')

In [ ]:
# Document-source resolver: corpus_expansion docs use targets_llama.jsonl + tokenized_manifest_llama.jsonl.
# Literary cells use MultilingualLiterary manifests at a different path.
# Buckeye uses a single JSONL of speaker-concatenated transcripts.
# Resolver yields (document_id, text) so the main loop is loader-agnostic.

# Load corpus_expansion sources.
tok_manifest = {}
if TOK_MANIFEST_PATH.exists():
    with open(TOK_MANIFEST_PATH) as f:
        for line in f:
            d = json.loads(line)
            tok_manifest[d['document_id']] = d['file_path']

all_targets = []
if TARGETS_PATH.exists():
    with open(TARGETS_PATH) as f:
        for line in f:
            all_targets.append(json.loads(line))
ce_corpora = {}
for t in all_targets:
    ce_corpora.setdefault(t['corpus_id'], set()).add(t['document_id'])

# MultilingualLiterary loader.
ML_DATA = DRIVE / 'Data/multilingual_literary'
def literary_docs(lang):
    """Yield (document_id, text) for MultilingualLiterary in given lang."""
    m = json.loads((ML_DATA / 'manifests' / f'{lang}.json').read_text(encoding='utf-8'))
    for author in m['authors']:
        for t in author['texts']:
            fp = ML_DATA / t['text_path']
            try:
                yield t['text_id'], fp.read_text(encoding='utf-8', errors='replace').strip()
            except Exception:
                continue

# Buckeye loader (spontaneous spoken English; speaker-concatenated jsonl).
BUCKEYE_JSONL = DRIVE / 'Data/buckeye_processed/speaker_concatenated.jsonl'
def buckeye_docs():
    """Yield (document_id, text) for Buckeye speaker-concatenated transcripts."""
    if not BUCKEYE_JSONL.exists():
        return
    with open(BUCKEYE_JSONL) as f:
        for line in f:
            d = json.loads(line)
            txt = d.get('text', '').strip()
            if txt:
                yield d['doc_id'], txt

def fix_ce_path(p):
    return str(p).replace('data/corpus_expansion/clean/',
        '/content/drive/MyDrive/LRTIA/Data/corpus_expansion/')

def ce_docs(corpus_id):
    """Yield (document_id, text) for corpus_expansion-format corpora."""
    for doc_id in ce_corpora.get(corpus_id, set()):
        fp = tok_manifest.get(doc_id)
        if fp is None: continue
        abs_fp = Path(fix_ce_path(fp))
        if not abs_fp.exists():
            abs_fp = Path('/content/drive/MyDrive/LRTIA/Data/corpus_expansion') / Path(fp).name
        try:
            yield doc_id, abs_fp.read_text(encoding='utf-8', errors='replace').strip()
        except Exception:
            continue

def get_doc_texts(corpus_id):
    """Unified resolver. Yields (document_id, text)."""
    if corpus_id == 'buckeye':
        yield from buckeye_docs()
    elif corpus_id.startswith('literary_'):
        lang = corpus_id.split('_', 1)[1]
        yield from literary_docs(lang)
    else:
        yield from ce_docs(corpus_id)

# Sanity preview.
for c in RUN_CORPORA:
    n = sum(1 for _ in get_doc_texts(c))
    print(f'  {c}: {n} docs found')

In [ ]:
# Main run loop.
for corpus_id in RUN_CORPORA:
    cache_path = BASE / f'{corpus_id}.json'
    if cache_path.exists():
        with open(cache_path) as f: n = len(json.load(f))
        print(f'\n{corpus_id}: cached ({n})'); continue

    docs = list(get_doc_texts(corpus_id))
    if not docs:
        print(f'\n{corpus_id}: no docs'); continue

    print(f'\n{"="*60}\n{corpus_id} ({len(docs)} candidate docs)\n{"="*60}')

    t0 = time.time()
    results = []
    skipped_short = 0

    for doc_id, text in tqdm(docs, desc=corpus_id):
        full_ids = tokenizer.encode(text, add_special_tokens=False)
        n_tok = len(full_ids)
        if n_tok < MIN_DOC_TOK:
            skipped_short += 1; continue

        rem_start = MAX_CTX
        rem_end = n_tok - TARGET_LEN
        for frac in TARGET_FRACS:
            ts = int(rem_start + frac * (rem_end - rem_start))
            te = ts + TARGET_LEN
            if ts - MAX_CTX < 0 or te > n_tok: continue

            r = compute_longrange_curves(full_ids, ts, te)
            r['corpus_id'] = corpus_id
            r['document_id'] = doc_id
            r['target_id'] = f'{doc_id}__pos{int(frac*100):02d}'
            r['target_frac'] = frac
            results.append(r)

    elapsed = time.time() - t0
    with open(cache_path, 'w') as f:
        json.dump(results, f)
    print(f'  {len(results)} results in {elapsed/60:.1f} min '
          f'({skipped_short} skipped: too short)')

    # Quick summary at the longest distance.
    if results:
        ord_long = float(np.mean([r['ordered_ppl'][-1] for r in results]))
        shuf_long = float(np.mean([r['shuffled_ppl'][-1] for r in results]))
        ord_zero = float(np.mean([r['ordered_ppl'][0] for r in results]))
        print(f'  ord_ppl[ctx=0]={ord_zero:.2f}  ord_ppl[ctx={MAX_CTX}]={ord_long:.2f}  '
              f'shuf_ppl[ctx={MAX_CTX}]={shuf_long:.2f}  gap@{MAX_CTX}={shuf_long-ord_long:+.2f}')

print('\nDone.')